<a href="https://colab.research.google.com/github/adrianwedd/afterwords/blob/main/colab/afterwords_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Afterwords — Voice Cloning ComparisonCompare **Qwen3-TTS 0.6B** and **1.7B** models side by side on a free Colab T4 GPU.| | 0.6B Base | 1.7B Base | 1.7B VoiceDesign ||---|---|---|---|| **Voice cloning** | Yes | Yes | No (text-described) || **VoiceDesign** | No | No | Yes || **T4 VRAM** | ~2 GB | ~5 GB | ~5 GB || **Used by Afterwords** | Yes (8-bit MLX) | — | — |**What you'll do:**1. Pick a voice from the 20 shipped profiles (or upload your own)2. Generate the same phrase with both models3. Compare quality side by side4. Try VoiceDesign — describe a voice in words, no audio needed5. Optionally batch-compare across all voices**Requirements:** T4 GPU runtime. Go to *Runtime → Change runtime type → T4 GPU*.---

## 1. Setup

In [ ]:
# ── Install dependencies ────────────────────────────────────────!pip install -q "qwen-tts>=0.1.1" soundfileimport torch, gc, os, json, timeimport soundfile as sffrom IPython.display import display, Audio, HTMLfrom google.colab import files# ── Verify GPU ─────────────────────────────────────────────────if not torch.cuda.is_available():    raise RuntimeError(        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU."    )# Auto-detect best dtype for this GPUDTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16gpu = torch.cuda.get_device_name(0)vram = torch.cuda.get_device_properties(0).total_memory / 1e9print(f"GPU: {gpu} ({vram:.1f} GB VRAM, dtype: {DTYPE})")print("Ready.")

## 2. Choose a VoiceSelect a shipped voice from the dropdown, or upload your own WAV file.

In [ ]:
# ── Clone repo for voice profiles ───────────────────────────────if not os.path.exists("afterwords"):    !git clone --depth 1 -q https://github.com/adrianwedd/afterwords.git    print("Cloned afterwords repo.")VOICES_DIR = "afterwords/voices"def load_voice_profiles(voices_dir):    """Load all voice profiles that have both a WAV and transcript."""    profiles = {}    for f in sorted(os.listdir(voices_dir)):        if not f.endswith(".json"):            continue        with open(os.path.join(voices_dir, f)) as fh:            try:                p = json.load(fh)            except json.JSONDecodeError:                continue        name = p.get("name", f.replace(".json", ""))        ref_wav = os.path.join(voices_dir, p.get("reference_audio", ""))        if os.path.exists(ref_wav) and p.get("reference_text"):            profiles[name] = {"audio": ref_wav, "text": p["reference_text"]}    return profilesprofiles = load_voice_profiles(VOICES_DIR)print(f"Loaded {len(profiles)} voices: {', '.join(profiles)}")

In [ ]:
# ── Pick a voice ───────────────────────────────────────────────## Change VOICE_NAME to any voice from the list above.# To upload your own, uncomment the "Option B" lines below.VOICE_NAME = "galadriel"  # @param {type:"string"}# Option B: Upload your own WAV# uploaded = files.upload()# VOICE_NAME = "custom"# profiles["custom"] = {#     "audio": list(uploaded.keys())[0],#     "text": "Type the exact transcript of your uploaded audio here.",# }# ── Text to synthesize ─────────────────────────────────────────SYNTH_TEXT = (    "You are absolutely right. "    "Your Claude Code session could sound like me.")# ── Verify ─────────────────────────────────────────────────────if VOICE_NAME not in profiles:    available = ", ".join(sorted(profiles))    raise ValueError(        f"Voice '{VOICE_NAME}' not found. Available: {available}"    )voice = profiles[VOICE_NAME]print(f"Voice:     {VOICE_NAME}")print(f"Ref text:  {voice['text'][:80]}...")print(f"Synth:     {SYNTH_TEXT}")print(f"\nReference audio:")display(Audio(voice["audio"]))

## 3. Generate & Compare

In [ ]:
# ── Shared functions ────────────────────────────────────────────def load_model(model_id):    """Load a Qwen3-TTS model, clearing GPU memory first."""    gc.collect()    torch.cuda.empty_cache()    print(f"Loading {model_id}...", end=" ")    t0 = time.time()    from qwen_tts import Qwen3TTSModel    model = Qwen3TTSModel.from_pretrained(        model_id, device_map="cuda:0", dtype=DTYPE,    )    vram = torch.cuda.memory_allocated() / 1e9    print(f"done ({time.time()-t0:.0f}s, {vram:.1f} GB VRAM)")    return modeldef clone_voice(model, text, ref_audio, ref_text):    """Generate speech by cloning a voice. Returns (waveform, sample_rate)."""    t0 = time.time()    wavs, sr = model.generate_voice_clone(        text=text,        language="English",        ref_audio=ref_audio,        ref_text=ref_text,        max_new_tokens=2048,        temperature=0.9,        top_p=1.0,        top_k=50,        repetition_penalty=1.05,    )    elapsed = time.time() - t0    duration = len(wavs[0]) / sr    rtf = elapsed / duration if duration > 0 else 0    print(f"  {duration:.1f}s audio in {elapsed:.1f}s (RTF={rtf:.2f}x)")    return wavs[0], srdef free_model(model_var_name):    """Delete a model and free GPU memory."""    if model_var_name in globals():        del globals()[model_var_name]    gc.collect()    torch.cuda.empty_cache()

In [ ]:
# ── 0.6B (same model Afterwords uses locally) ─────────────────model_06b = load_model("Qwen/Qwen3-TTS-12Hz-0.6B-Base")print(f"Synthesizing '{SYNTH_TEXT[:40]}...' with 0.6B:")wav_06b, sr_06b = clone_voice(    model_06b, SYNTH_TEXT, voice["audio"], voice["text"])sf.write("output_06b.wav", wav_06b, sr_06b)free_model("model_06b")display(Audio("output_06b.wav"))

In [ ]:
# ── 1.7B (larger, higher quality) ──────────────────────────────model_17b = load_model("Qwen/Qwen3-TTS-12Hz-1.7B-Base")print(f"Synthesizing '{SYNTH_TEXT[:40]}...' with 1.7B:")wav_17b, sr_17b = clone_voice(    model_17b, SYNTH_TEXT, voice["audio"], voice["text"])sf.write("output_17b.wav", wav_17b, sr_17b)free_model("model_17b")display(Audio("output_17b.wav"))

## 4. Side-by-Side Comparison

In [ ]:
display(HTML(f"""<div style="font-family: system-ui; background: #1a1a2e; padding: 20px;            border-radius: 12px; color: #e0e0e0;">  <h3 style="margin:0 0 4px;">🎙 {VOICE_NAME}</h3>  <p style="color:#999; margin:0 0 16px; font-style:italic;">"{SYNTH_TEXT}"</p>  <table style="width:100%; border-collapse:collapse;">    <tr>      <td style="padding:8px 12px; background:#16213e; border-radius:8px 0 0 8px; width:50%;">        <strong>0.6B</strong> <span style="color:#888">(Afterwords default)</span>      </td>      <td style="padding:8px 12px; background:#1a1a3e; border-radius:0 8px 8px 0; width:50%;">        <strong>1.7B</strong> <span style="color:#888">(larger model)</span>      </td>    </tr>  </table></div>"""))print("\n0.6B:")display(Audio("output_06b.wav"))print("\n1.7B:")display(Audio("output_17b.wav"))

## 5. VoiceDesign (1.7B only)The 1.7B VoiceDesign model creates voices from **text descriptions** — no reference audio needed. Describe the voice you want in plain English.

In [ ]:
# ── Describe the voice you want ─────────────────────────────────VOICE_DESCRIPTION = (    "Speak in a deep, theatrical British voice with a hint of "    "mischief, like a Shakespearean villain who finds everything "    "secretly amusing.")# Try these alternatives by uncommenting:# VOICE_DESCRIPTION = "A warm, gentle grandmother reading a bedtime story."# VOICE_DESCRIPTION = "Speak in an incredulous tone, with panic creeping into your voice."# VOICE_DESCRIPTION = "A calm, measured newsreader with a slight Australian accent."# VOICE_DESCRIPTION = "An excited sports commentator calling a last-second goal."# VOICE_DESCRIPTION = "A tired, world-weary detective narrating a case."DESIGN_TEXT = (    "I must say, your code is quite remarkable. "    "Almost as remarkable as mine.")# ── Generate ───────────────────────────────────────────────────model_vd = load_model("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign")print(f"Voice: {VOICE_DESCRIPTION[:60]}...")print(f"Text:  {DESIGN_TEXT}")t0 = time.time()wavs, sr = model_vd.generate_voice_design(    text=DESIGN_TEXT,    language="English",    instruct=VOICE_DESCRIPTION,    max_new_tokens=2048,)elapsed = time.time() - t0duration = len(wavs[0]) / srprint(f"\nGenerated {duration:.1f}s audio in {elapsed:.1f}s")sf.write("output_voice_design.wav", wavs[0], sr)free_model("model_vd")display(Audio("output_voice_design.wav"))

## 6. Batch Comparison (All Voices)Run the same phrase through **every shipped voice** with both models. Takes ~10-20 minutes depending on voice count. Skip this if you just wanted a quick test.

In [ ]:
BATCH_TEXT = "You are absolutely right. Your Claude Code session could sound like me."os.makedirs("comparison", exist_ok=True)results = {}for model_label, model_id in [    ("0.6B", "Qwen/Qwen3-TTS-12Hz-0.6B-Base"),    ("1.7B", "Qwen/Qwen3-TTS-12Hz-1.7B-Base"),]:    model = load_model(model_id)    results[model_label] = {}    for i, (name, v) in enumerate(profiles.items(), 1):        print(f"  [{i}/{len(profiles)}] {model_label} × {name}:", end="")        try:            wav, sr = clone_voice(model, BATCH_TEXT, v["audio"], v["text"])            out_path = f"comparison/{name}_{model_label}.wav"            sf.write(out_path, wav, sr)            results[model_label][name] = out_path        except Exception as e:            print(f"  FAILED: {e}")            results[model_label][name] = None    free_model("model")print(f"\nDone — {sum(1 for v in results['0.6B'].values() if v)} + "      f"{sum(1 for v in results['1.7B'].values() if v)} files generated.")

In [ ]:
# ── Listen to batch results ─────────────────────────────────────for name in profiles:    path_06b = results.get("0.6B", {}).get(name)    path_17b = results.get("1.7B", {}).get(name)    if not path_06b and not path_17b:        continue    display(HTML(f"<h4 style='margin:16px 0 4px;'>{name}</h4>"))    if path_06b:        print("0.6B:")        display(Audio(path_06b))    if path_17b:        print("1.7B:")        display(Audio(path_17b))

## 7. Download Results

In [ ]:
import shutil, globwavs = glob.glob("output_*.wav") + glob.glob("comparison/*.wav")if wavs:    os.makedirs("download", exist_ok=True)    for w in wavs:        shutil.copy(w, "download/")    shutil.make_archive("afterwords_comparison", "zip", "download")    print(f"Packed {len(wavs)} files into afterwords_comparison.zip")    files.download("afterwords_comparison.zip")else:    print("No audio files generated yet. Run the cells above first.")

In [ ]:
# ── Optional: Save to Google Drive ──────────────────────────────# Uncomment to mount Drive and copy the zip there.# from google.colab import drive# drive.mount("/content/drive")# DRIVE_FOLDER = "/content/drive/MyDrive/Afterwords/"# os.makedirs(DRIVE_FOLDER, exist_ok=True)# if os.path.exists("afterwords_comparison.zip"):#     shutil.copy("afterwords_comparison.zip", DRIVE_FOLDER)#     print(f"Saved to {DRIVE_FOLDER}")

---### Notes- **Local use:** Afterwords runs the 0.6B model (8-bit quantized via MLX) on Apple Silicon Macs. See [github.com/adrianwedd/afterwords](https://github.com/adrianwedd/afterwords).- **This notebook** uses the official PyTorch weights on CUDA — different backend, same voices.- **VoiceDesign** is exclusive to the 1.7B VoiceDesign variant — it creates voices from text descriptions, no reference audio needed.- **Voice cloning quality** depends heavily on the reference clip: clean solo speech with an accurate transcript produces the best results.- All models by [Qwen/Alibaba](https://github.com/QwenLM/Qwen3-TTS) under Apache 2.0 license.